In [ ]:
!pip install -q pymupdf ollama openai

In [ ]:
import os
import re
import unicodedata
import pymupdf
import ollama

from openai import OpenAI
from getpass import getpass
from google.colab import files

# Upload PDF
uploaded = files.upload()

PDF_PATH = next(iter(uploaded))

print(f"Selected PDF: {PDF_PATH}")

In [ ]:
# ==========================================
# Settings
# ==========================================

OUTPUT_PATH = "summary.txt"
EXTRACTED_TEXT_PATH = "extracted_text.txt"

# Local AI
MODEL_NAME = "qwen2.5:3b"

# Online AI
ONLINE_MODEL_NAME = "openai/gpt-oss-120b"

# Processing
CHUNK_SIZE = 3500

# AI Status
LOCAL_OLLAMA_AVAILABLE = None

# ==========================================
# Groq API
# ==========================================

GROQ_API_KEY = getpass("Enter your new Groq API Key: ")

online_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

print("Settings initialized successfully.")

In [ ]:
# ==========================================
# PDF Text Extraction
# ==========================================

def get_text_direction(text):
    """Determine whether the text is primarily Arabic or English."""

    arabic_count = 0
    english_count = 0

    for char in text:
        if "\u0600" <= char <= "\u06FF":
            arabic_count += 1
        elif "a" <= char.lower() <= "z":
            english_count += 1

    if arabic_count > english_count:
        return "rtl"

    return "ltr"


def extract_page_text(page):

    words = page.get_text("words")

    if not words:
        return ""

    lines = []

    # Group words according to their vertical positions.
    for word in words:

        x0, y0, x1, y1, text, block_no, line_no, word_no = word

        matched_line = None

        for line in lines:

            if abs(line["y"] - y0) <= 3:
                matched_line = line
                break

        if matched_line is None:

            matched_line = {
                "y": y0,
                "words": []
            }

            lines.append(matched_line)

        matched_line["words"].append((x0, text))

    # Sort lines from top to bottom.
    lines.sort(key=lambda line: line["y"])

    result = []

    for line in lines:

        original_line = " ".join(
            text for _, text in line["words"]
        )

        direction = get_text_direction(original_line)

        # Arabic: right to left. English: left to right.
        if direction == "rtl":

            line["words"].sort(
                key=lambda item: item[0],
                reverse=True
            )

        else:

            line["words"].sort(
                key=lambda item: item[0]
            )

        line_text = " ".join(
            word_text for _, word_text in line["words"]
        )

        result.append(line_text)

    return "\n".join(result)


def clean_text(text):
    """Keep Arabic letters, English letters, numbers, %, and whitespace."""

    text = re.sub(
        r"[^a-zA-Z0-9\u0621-\u064A\u0660-\u0669%\s]",
        "",
        text
    )

    # Normalize spaces.
    text = re.sub(r"\s+", " ", text)

    # Format percentages: 60 % -> 60%.
    text = re.sub(r"(\d+)\s+%", r"\1%", text)

    return text.strip()


def extract_pdf_text(pdf_path):
    """Extract and clean text from a PDF."""

    pages_text = []

    with pymupdf.open(pdf_path) as document:

        if len(document) == 0:
            raise ValueError("The PDF file is empty.")

        for page in document:

            extracted_text = extract_page_text(page)

            cleaned_text = clean_text(extracted_text)

            if cleaned_text:
                pages_text.append(cleaned_text)

    return "\n\n".join(pages_text)

In [ ]:
# ==========================================
# Summarization Instructions
# ==========================================

SYSTEM_PROMPT = """
You are a strict and faithful document summarizer.

Your highest priority is factual accuracy and faithfulness
to the supplied source text.

Follow these rules strictly:

1. Use ONLY information explicitly stated in the supplied text.
2. Never invent facts, explanations, examples, causes, or conclusions.
3. Never use your external knowledge to fill in missing information.
4. Preserve the original meaning and context of every statement.
5. Preserve important names, organizations, dates, numbers, percentages,
   quantities, and other factual details accurately.
6. Preserve uncertainty. Words such as "may", "might", "could",
   "possibly", and their equivalents must not become certain claims.
7. Preserve attribution. Do not attribute a statement, warning,
   recommendation, or opinion to an organization unless the source does so.
8. Do not introduce causal relationships or conclusions absent from the source.
9. Do not remove qualifications that are necessary for factual accuracy.
10. Remove repetition and unimportant details only when doing so
    does not change the meaning.
11. Use the same language as the source text.
12. If the source is Arabic, write the summary in Arabic.
    If the source is English, write the summary in English.
13. If a detail is unclear or unsupported, omit it rather than guess.
14. Do not add personal opinions, introductions, or external explanations.
15. Return only the summary.

Accuracy is more important than making the summary shorter.
Do not claim anything that cannot be supported by the source text.
"""


# ==========================================
# Split Text
# ==========================================

def split_text(text, chunk_size=CHUNK_SIZE):

    paragraphs = text.splitlines()

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        paragraph = paragraph.strip()

        if not paragraph:
            continue

        # Split exceptionally long paragraphs into smaller pieces.
        if len(paragraph) > chunk_size:

            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""

            for start in range(0, len(paragraph), chunk_size):

                piece = paragraph[start:start + chunk_size]

                chunks.append(piece)

            continue

        if len(current_chunk) + len(paragraph) + 1 <= chunk_size:

            if current_chunk:
                current_chunk += "\n" + paragraph

            else:
                current_chunk = paragraph

        else:

            if current_chunk:
                chunks.append(current_chunk)

            current_chunk = paragraph

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [ ]:
# ==========================================
# AI Provider
# ==========================================

def call_ollama(system_prompt, user_prompt):

    global LOCAL_OLLAMA_AVAILABLE
    global online_client

    # ======================================
    # Try Local Ollama
    # ======================================

    if LOCAL_OLLAMA_AVAILABLE is not False:

        try:

            print("[AI] Trying Local Ollama...")

            response = ollama.chat(

                model=MODEL_NAME,

                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],

                options={
                    "temperature": 0,
                    "num_ctx": 4096
                }
            )

            result = response["message"]["content"].strip()

            if not result:
                raise ValueError("Empty Ollama response")

            LOCAL_OLLAMA_AVAILABLE = True

            print("[AI] Using Local Ollama")

            return result

        except Exception as error:

            print("\n[AI] Local Ollama is unavailable.")
            print(f"[AI] Reason: {error}")

            LOCAL_OLLAMA_AVAILABLE = False

    # ======================================
    # Fallback to Online AI
    # ======================================

    print("\n[AI] Switching to Online AI...")

    try:

        response = online_client.chat.completions.create(

            model=ONLINE_MODEL_NAME,

            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],

            temperature=0
        )

        result = response.choices[0].message.content

        if not result or not result.strip():

            raise ValueError("Empty Online AI response")

        print("[AI] Using Online Groq API")

        return result.strip()

    except Exception as error:

        raise RuntimeError(
            f"Both AI providers failed.\n"
            f"Online AI Error: {error}"
        )

In [ ]:
# ==========================================
# Summarize One Chunk
# ==========================================

def summarize_chunk(text):

    user_prompt = (
        "Summarize the following source text.\n\n"
        "Requirements:\n"
        "- Include only facts supported by this text.\n"
        "- Preserve important numbers, names, dates, and attributions.\n"
        "- Preserve uncertainty and qualifications.\n"
        "- Do not add external knowledge or unsupported conclusions.\n"
        "- Do not change the meaning of the source.\n"
        "- Use the same language as the source.\n"
        "- Return only the summary.\n\n"
        "SOURCE TEXT:\n"
        "----- BEGIN SOURCE -----\n"
        + text
        + "\n----- END SOURCE -----"
    )

    return call_ollama(
        SYSTEM_PROMPT,
        user_prompt
    )

In [ ]:
# ==========================================
# Create Final Summary
# ==========================================

def create_final_summary(original_text, partial_summaries):

    """
    Generate the final summary using the original source text
    as the primary reference, not just the partial summaries.
    """

    partial_text = "\n\n".join(

        f"Partial summary {index}:\n{summary}"

        for index, summary in enumerate(partial_summaries, start=1)

    )

    # If the original document fits within the model's context,
    # summarize it directly for better factual consistency.

    if len(original_text) <= 7000:

        user_prompt = (

            "Create a faithful summary of the original source text below.\n\n"

            "The original source is the authority. Any partial summaries "
            "are secondary and may contain omissions or errors.\n\n"

            "Rules:\n"
            "1. Use only facts explicitly supported by the original source.\n"
            "2. Do not copy a claim from a partial summary unless it is "
            "supported by the original source.\n"
            "3. Preserve numbers, names, dates, attributions, and uncertainty.\n"
            "4. Do not introduce new facts, interpretations, or conclusions.\n"
            "5. Keep the summary clear and concise without losing essential facts.\n"
            "6. Return only the summary.\n\n"

            "ORIGINAL SOURCE TEXT:\n"
            "----- BEGIN SOURCE -----\n"
            + original_text
            + "\n----- END SOURCE -----\n\n"

            "PARTIAL SUMMARIES FOR REFERENCE ONLY:\n"
            + partial_text

        )

        return call_ollama(
            SYSTEM_PROMPT,
            user_prompt
        )

    # For longer documents, combine partial summaries carefully.

    user_prompt = (

        "Create one coherent final summary from the partial summaries below.\n\n"

        "Strict rules:\n"
        "1. Use only information contained in the partial summaries.\n"
        "2. Do not add external knowledge or new facts.\n"
        "3. Do not invent missing details or complete incomplete statements.\n"
        "4. Preserve numbers, names, dates, attributions, and uncertainty.\n"
        "5. Do not turn possibilities into certainties.\n"
        "6. Remove repetition without changing the meaning.\n"
        "7. If two statements appear inconsistent, do not invent a resolution.\n"
        "8. Return only the final summary.\n\n"

        "PARTIAL SUMMARIES:\n"
        "----- BEGIN SUMMARIES -----\n"
        + partial_text
        + "\n----- END SUMMARIES -----"

    )

    return call_ollama(
        SYSTEM_PROMPT,
        user_prompt
    )

In [ ]:
# ==========================================
# Summarize PDF
# ==========================================

def summarize_pdf(pdf_path):

    print("\nExtracting PDF text...")

    if not os.path.exists(pdf_path):

        raise FileNotFoundError(
            f"PDF file not found: {pdf_path}"
        )

    # Extract text using the existing extraction function.
    text = extract_pdf_text(pdf_path)

    if not text.strip():

        raise ValueError(
            "No text was extracted from the PDF."
        )

    print(f"Extracted characters: {len(text)}")

    # Save extracted text.
    with open(
        EXTRACTED_TEXT_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(text)

    print(
        f"Extracted text saved to: "
        f"{os.path.abspath(EXTRACTED_TEXT_PATH)}"
    )

    # Split the text into chunks.
    chunks = split_text(text)

    if not chunks:

        raise ValueError(
            "The extracted text could not be split into chunks."
        )

    print(f"Number of chunks: {len(chunks)}")

    summaries = []

    # Summarize each chunk.
    for index, chunk in enumerate(chunks, start=1):

        print(
            f"Summarizing chunk {index}/{len(chunks)}..."
        )

        summary = summarize_chunk(chunk)

        if summary:
            summaries.append(summary)

    if not summaries:

        raise ValueError(
            "The model did not generate a summary."
        )

    # A single chunk does not require a combining step.
    if len(summaries) == 1:

        final_summary = summaries[0]

    else:

        print("\nCreating final summary...")

        final_summary = create_final_summary(
            text,
            summaries
        )

    if not final_summary.strip():

        raise ValueError(
            "The final summary is empty."
        )

    return final_summary

In [ ]:
# ==========================================
# Main
# ==========================================

try:

    print("Initializing AI Providers...")

    print(f"Local Model: {MODEL_NAME}")
    print(f"Online Model: {ONLINE_MODEL_NAME}")

    summary = summarize_pdf(PDF_PATH)

    # Save final summary.
    with open(
        OUTPUT_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(summary)

    print("\n" + "=" * 50)
    print("FINAL SUMMARY")
    print("=" * 50)

    print(summary)

    print("\n" + "=" * 50)

    print(
        f"Summary saved to: "
        f"{os.path.abspath(OUTPUT_PATH)}"
    )

except FileNotFoundError as error:

    print(f"\nFile error: {error}")

except ValueError as error:

    print(f"\nData error: {error}")

except Exception as error:

    print("\nAn error occurred:")
    print(f"{type(error).__name__}: {error}")

In [ ]:
# Download summary
files.download(OUTPUT_PATH)

# Download extracted text
files.download(EXTRACTED_TEXT_PATH)